# Obiettivo: Scraping + Salvataggio dati grezzi



In [29]:
# SETUP AND LIBRARY INSTALLATION

# Install necessary libraries
import sys
import subprocess

def install_package(package):
    """Installs Python package if not present."""
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

# Install dependencies
packages = ['requests', 'beautifulsoup4', 'pandas', 'regex']
for pkg in packages:
    install_package(pkg)

print("Libraries installed successfully\n")

# Import libraries
import json
import time
import random
import requests
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup

print("Imports completed")

Installing beautifulsoup4...
Libraries installed successfully

Imports completed


In [2]:
# GOOGLE DRIVE SETUP

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("\nGoogle Drive Check")

# Create project folder
import os

project_folder = '/content/drive/MyDrive/DataManagement_Project'
data_folder = os.path.join(project_folder, 'data')
logs_folder = os.path.join(project_folder, 'logs')
checkpoint_folder = os.path.join(project_folder, 'checkpoints')

# Create folders if they don't exist
for folder in [project_folder, data_folder, logs_folder, checkpoint_folder]:
    if not os.path.exists(folder):
        os.makedirs(folder)
        print(f"Folder created: {folder}")
    else:
        print(f"Folder exists: {folder}")

print("\nFolder structure:")
print(f"  {project_folder}/")
print(f"    ├── data/          (JSON files)")
print(f"    ├── logs/          (scraping logs)")
print(f"    └── checkpoints/   (progress checkpoints)")

Mounted at /content/drive

Google Drive Check
Folder exists: /content/drive/MyDrive/DataManagement_Project
Folder exists: /content/drive/MyDrive/DataManagement_Project/data
Folder exists: /content/drive/MyDrive/DataManagement_Project/logs
Folder exists: /content/drive/MyDrive/DataManagement_Project/checkpoints

Folder structure:
  /content/drive/MyDrive/DataManagement_Project/
    ├── data/          (JSON files)
    ├── logs/          (scraping logs)
    └── checkpoints/   (progress checkpoints)


In [32]:
# SOURCE 1: WEB SCRAPING AUTOSCOUT24 LISTINGS

import re
import requests
import random
import os
import json
import time
from datetime import datetime
from bs4 import BeautifulSoup

# USER AGENTS
USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/109.0.2227.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/109.0.3497.92 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_0) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36',
]

def create_session():
    """Creates HTTP session with realistic headers"""
    session = requests.Session()
    session.headers.update({
        "User-Agent": random.choice(USER_AGENTS),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1"
    })
    return session


def scrape_overnight_safe(start_url, target_count=2000, checkpoint_every=10):
    """
    Overnight scraping with Google Drive saving

    Args:
        start_url: Initial URL
        target_count: Target number of listings
        checkpoint_every: Save every N pages

    Returns:
        list: All collected listings
    """

    print("\n" + "="*70)
    print("OVERNIGHT SCRAPING")
    print("="*70)
    print(f"\nTarget: {target_count} listings")
    print(f"Strategy: Long pauses (20-40s) between pages")
    print(f"Checkpoint: Saving every {checkpoint_every} pages")
    print(f"Storage: Google Drive")

    # Calculate needed pages
    pages_needed = (target_count // 20) + 5
    print(f"\nPages to scrape: approx {pages_needed}")
    print(f"Estimated time: {pages_needed * 30 / 60:.1f} minutes")

    # Checkpoint file on Google Drive
    checkpoint_file = os.path.join(checkpoint_folder, 'scraping_checkpoint.json')

    # Check for previous checkpoint
    if os.path.exists(checkpoint_file):
        print(f"\n[CHECKPOINT] Previous checkpoint found")
        with open(checkpoint_file, 'r', encoding='utf-8') as f:
            checkpoint_data = json.load(f)

        all_listings = checkpoint_data['listings']
        start_page = checkpoint_data['last_page'] + 1

        print(f"             Resuming from page {start_page}")
        print(f"             Listings already collected: {len(all_listings)}")

        user_input = input("\nDo you want to resume from here? (y/n): ").lower()
        if user_input != 'y':
            print("Restarting from scratch...\n")
            all_listings = []
            start_page = 1
    else:
        all_listings = []
        start_page = 1

    # Session
    session = create_session()
    search_id = None

    # Log file on Google Drive
    log_filename = f'scraping_log_{datetime.now().strftime("%Y%m%d_%H%M%S")}.txt'
    log_file = os.path.join(logs_folder, log_filename)

    def log(message):
        """Writes to console and file"""
        timestamp = datetime.now().strftime("%H:%M:%S")
        log_message = f"[{timestamp}] {message}"
        print(log_message)
        with open(log_file, 'a', encoding='utf-8') as f:
            f.write(log_message + '\n')

    log(f"Starting scraping from page {start_page}")
    log(f"Checkpoint file: {checkpoint_file}")
    log(f"Log file: {log_file}")

    page = start_page
    consecutive_empty = 0

    while len(all_listings) < target_count:
        # Construct URL
        if page == 1:
            url = start_url
        else:
            if '?' in start_url:
                url = f"{start_url}&page={page}"
            else:
                url = f"{start_url}?page={page}"

            if search_id:
                url += f"&search_id={search_id}"
            url += "&source=listpage_pagination"

        log(f"\n{'='*70}")
        log(f"PAGE {page}")
        log(f"Listings collected: {len(all_listings)}/{target_count}")
        log(f"Remaining: {target_count - len(all_listings)}")
        log(f"{'='*70}")
        log(f"URL: {url[:100]}...")

        try:
            # LONG PAUSE
            if page > start_page:
                sleep_time = random.uniform(20, 40)
                log(f"Pausing for {sleep_time:.1f}s...")
                time.sleep(sleep_time)

            # Request
            response = session.get(url, timeout=30)
            final_url = response.url

            # Extract search_id
            if page == 1 and 'search_id=' in final_url:
                match = re.search(r'search_id=([^&]+)', final_url)
                if match:
                    search_id = match.group(1)
                    log(f"Search ID extracted: {search_id}")

            # Check status
            if response.status_code == 403:
                log("ERROR: Blocked (403 Forbidden)")
                log("Pausing 1 minute and retrying...")
                time.sleep(60)
                session = create_session()
                continue

            if response.status_code != 200:
                log(f"ERROR: Status {response.status_code}")
                page += 1
                continue

            log(f"Response OK (status: {response.status_code})")

            # Parse HTML
            soup = BeautifulSoup(response.content, 'html.parser')
            car_articles = soup.find_all('article', class_='cldt-summary-full-item')

            log(f"Listings found: {len(car_articles)}")

            if len(car_articles) == 0:
                consecutive_empty += 1
                log(f"Empty page ({consecutive_empty} consecutive)")

                if consecutive_empty >= 3:
                    log("3 empty pages - ending pagination")
                    break

                page += 1
                continue
            else:
                consecutive_empty = 0

            # Extract listings
            page_listings = []
            for idx, article in enumerate(car_articles, 1):
                try:
                    # TITLE
                    title = None
                    make = article.get('data-make', '')
                    model = article.get('data-model', '')
                    if make and model:
                      title = f"{make.capitalize()} {model.capitalize()}"

                    # DATA FROM DATA-* ATTRIBUTES
                    price_raw = article.get('data-price')
                    mileage_raw = article.get('data-mileage')
                    year = article.get('data-first-registration')
                    fuel_code = article.get('data-fuel-type')

                    # English Mapping
                    fuel_map = {'d': 'Diesel', 'b': 'Gasoline', 'e': 'Electric',
                                'h': 'Hybrid', 'c': 'CNG', 'l': 'LPG', '2': 'Hybrid'}
                    fuel_type = fuel_map.get(fuel_code, fuel_code)

                    # POWER: Search in pills first
                    power = None

                    # Search in pills
                    vehicle_details_div = article.find('div', class_='DeclutteredListItem_vehicleDetails__MQjEf')
                    if vehicle_details_div:
                        pill_texts = vehicle_details_div.find_all('span', class_='ListItemPill_text__Cr6mq')
                        for pill in pill_texts:
                            text = pill.get_text().strip()
                            if 'kW' in text or 'hp' in text:
                                power = text
                                break


                    # SELLER
                    seller_type_code = article.get('data-seller-type')
                    seller_type_map = {'d': 'Dealer', 'p': 'Private'}
                    seller_type = seller_type_map.get(seller_type_code, seller_type_code)

                    seller_name_elem = article.find('span', class_='ListItemSeller_name__3T6DT')
                    seller_name = seller_name_elem.get_text().strip() if seller_name_elem else None

                    zip_code = article.get('data-listing-zip-code')
                    country_code = article.get('data-listing-country')


                    # LINK - Constructed from data-guid
                    link = None
                    guid = article.get('data-guid')

                    if guid:
                        make_attr = article.get('data-make', '')
                        model_attr = article.get('data-model', '')

                        # Clean and format for URL
                        make_clean = make_attr.replace(' ', '-').replace('_', '-').lower() if make_attr else 'car'
                        model_clean = model_attr.replace(' ', '-').replace('_', '-').lower() if model_attr else 'model'

                        # Construct standard AutoScout24 URL
                        link = f"https://www.autoscout24.com/offers/{make_clean}-{model_clean}-{guid}"



                    # CREATE LISTING
                    listing = {
                        'id': article.get('data-guid'),
                        'title': title,
                        'price': f"EUR {price_raw}" if price_raw else None,
                        'price_raw': int(price_raw) if price_raw and price_raw.isdigit() else None,
                        'make': article.get('data-make'),
                        'model': article.get('data-model'),
                        'mileage': f"{mileage_raw} km" if mileage_raw else None,
                        'mileage_raw': int(mileage_raw) if mileage_raw and mileage_raw.isdigit() else None,
                        'year': year,
                        'fuel_type': fuel_type,
                        'fuel_type_code': fuel_code,
                        'power': power,
                        'seller_type': seller_type,
                        'seller_name': seller_name,
                        'seller_zip': zip_code,
                        'seller_country': country_code,
                        'link': link,
                        'page': page,
                        'source': 'autoscout24_scraping',
                        'scrape_date': datetime.now().isoformat()
                    }

                    if listing['title'] or (listing['make'] and listing['model']):
                        page_listings.append(listing)

                except Exception as e:
                    log(f"Error listing {idx}: {e}")
                    continue

            log(f"Extracted: {len(page_listings)} listings")
            all_listings.extend(page_listings)

            # CHECKPOINT on Google Drive
            if page % checkpoint_every == 0:
                log(f"\n[CHECKPOINT] Saving to Google Drive...")
                checkpoint_data = {
                    'last_page': page,
                    'last_update': datetime.now().isoformat(),
                    'total_listings': len(all_listings),
                    'search_id': search_id,
                    'listings': all_listings
                }

                with open(checkpoint_file, 'w', encoding='utf-8') as f:
                    json.dump(checkpoint_data, f, ensure_ascii=False, indent=2)

                log(f"             Checkpoint saved: {len(all_listings)} listings")
                log(f"             File: {checkpoint_file}")

            page += 1

        except KeyboardInterrupt:
            log("\n\n[INTERRUPTION] CTRL+C received")
            log(f"Listings collected: {len(all_listings)}")
            log("Saving checkpoint to Google Drive...")

            checkpoint_data = {
                'last_page': page,
                'last_update': datetime.now().isoformat(),
                'total_listings': len(all_listings),
                'search_id': search_id,
                'listings': all_listings
            }

            with open(checkpoint_file, 'w', encoding='utf-8') as f:
                json.dump(checkpoint_data, f, ensure_ascii=False, indent=2)

            log(f"Checkpoint saved: {checkpoint_file}")
            log("You can resume by re-running the script")
            break

        except Exception as e:
            log(f"ERROR: {e}")
            log("Continuing with next page...")
            page += 1
            time.sleep(30)
            continue

    # Remove checkpoint if completed
    if len(all_listings) >= target_count and os.path.exists(checkpoint_file):
        os.remove(checkpoint_file)
        log("\nCheckpoint removed (scraping completed)")

    log(f"\n{'='*70}")
    log("SCRAPING COMPLETED")
    log(f"{'='*70}")
    log(f"Total listings: {len(all_listings)}")
    log(f"Pages scraped: {page - start_page + 1}")
    log(f"Log saved: {log_file}")

    return all_listings


# EXECUTION
print("\n[START] Overnight Scraping Mode with Google Drive")
print("\nFEATURES:")
print("- Estimated time: 60-90 minutes for 2000 listings")
print("- Automatic checkpoints every 10 pages")
print("- All files saved to Google Drive")
print("- Interruption with CTRL+C saves progress")

start_url = "https://www.autoscout24.com/lst?atype=C"

listings_data = scrape_overnight_safe(
    start_url=start_url,
    target_count=8000,
    checkpoint_every=10
)

# Remove duplicates
seen_ids = set()
unique_listings = []

for listing in listings_data:
    listing_id = listing.get('id')
    if listing_id and listing_id not in seen_ids:
        seen_ids.add(listing_id)
        unique_listings.append(listing)
    elif not listing_id:
        unique_listings.append(listing)

# Final JSON
scraping_result = {
    'source': 'autoscout24_website',
    'collection_date': datetime.now().isoformat(),
    'method': 'requests + beautifulsoup4',
    'strategy': 'overnight_slow_scraping',
    'target_count': 8000,
    'total_listings': len(unique_listings),
    'duplicates_removed': len(listings_data) - len(unique_listings),
    'listings': unique_listings
}

# SAVE to Google Drive
output_file_1 = os.path.join(data_folder, 'autoscout24_listings.json')
print(f"\n[SAVE] Saving final file to Google Drive...")
print(f"       Path: {output_file_1}")

with open(output_file_1, 'w', encoding='utf-8') as f:
    json.dump(scraping_result, f, ensure_ascii=False, indent=2)

print(f"       File saved successfully")
print(f"\n[STATS] FINAL Summary:")
print(f"        - Total listings: {scraping_result['total_listings']}")
print(f"        - Duplicates removed: {scraping_result['duplicates_removed']}")
print(f"        - JSON File: {output_file_1}")


[START] Overnight Scraping Mode with Google Drive

FEATURES:
- Estimated time: 60-90 minutes for 2000 listings
- Automatic checkpoints every 10 pages
- All files saved to Google Drive
- Interruption with CTRL+C saves progress

OVERNIGHT SCRAPING

Target: 8000 listings
Strategy: Long pauses (20-40s) between pages
Checkpoint: Saving every 10 pages
Storage: Google Drive

Pages to scrape: approx 405
Estimated time: 202.5 minutes

[CHECKPOINT] Previous checkpoint found
             Resuming from page 5
             Listings already collected: 59

Do you want to resume from here? (y/n): n
Restarting from scratch...

[10:50:51] Starting scraping from page 1
[10:50:51] Checkpoint file: /content/drive/MyDrive/DataManagement_Project/checkpoints/scraping_checkpoint.json
[10:50:51] Log file: /content/drive/MyDrive/DataManagement_Project/logs/scraping_log_20251229_105051.txt
[10:50:51] 
[10:50:51] PAGE 1
[10:50:51] Listings collected: 0/8000
[10:50:51] Remaining: 8000
[10:50:51] ==================

# SCRAPING DETAILS

In [34]:
# SOURCE 2: CAR DETAILS SCRAPER

import re
import time
import random
import json
import os
import requests
from datetime import datetime
from bs4 import BeautifulSoup

# ============================================
# SETUP & CONFIG
# ============================================
# Assumes 'checkpoint_folder', 'logs_folder', 'data_folder',
# and 'create_session()' are defined in the environment.

print("\n" + "="*70)
print("SOURCE 2: CAR DETAILS SCRAPER (ROBUST MODE)")
print("="*70)

def extract_guid_from_url(url):
    """Extracts the UUID/GUID from the AutoScout24 URL."""
    try:
        # The GUID is usually the last part of the URL segments
        # Format: .../make-model-version-GUID
        clean_url = url.split('?')[0].rstrip('/')
        last_part = clean_url.split('/')[-1]
        segments = last_part.split('-')

        # Valid GUIDs usually have 5 segments when split by '-'
        if len(segments) >= 5:
            return '-'.join(segments[-5:])
        return None
    except Exception:
        return None

def scrape_car_details(session, url):
    """Scrapes detailed data from a single listing page."""
    try:
        # Human-like pause
        time.sleep(random.uniform(5, 10))

        response = session.get(url, timeout=30)
        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.content, 'html.parser')
        data = {}

        # ID & URL
        data['id'] = extract_guid_from_url(url)
        data['url'] = url

        # --- BASIC INFO ---
        # Regex used to match classes regardless of hashed suffixes (e.g., StageTitle_title__xyz)
        title_node = soup.find('h1', class_=re.compile(r'StageTitle_title'))
        if title_node:
            make_model = title_node.find('span', class_=re.compile(r'StageTitle_boldClassifiedInfo'))
            version = title_node.find('div', class_=re.compile(r'StageTitle_modelVersion'))

            if make_model and version:
                data['title'] = f"{make_model.get_text().strip()} {version.get_text().strip()}"
            elif make_model:
                data['title'] = make_model.get_text().strip()
            else:
                data['title'] = title_node.get_text().strip()

        price_node = soup.find('span', class_=re.compile(r'PriceInfo_price'))
        if price_node:
            data['price'] = price_node.get_text().strip()

        # --- VEHICLE OVERVIEW (Icons section) ---
        data['specifications'] = {}
        overview_items = soup.find_all('div', class_=re.compile(r'VehicleOverview_itemContainer'))

        for item in overview_items:
            t_node = item.find('div', class_=re.compile(r'VehicleOverview_itemTitle'))
            v_node = item.find('div', class_=re.compile(r'VehicleOverview_itemText'))

            if t_node and v_node:
                key = t_node.get_text().strip()
                val = v_node.get_text().strip()

                # Basic mapping
                if 'Mileage' in key: data['specifications']['mileage'] = val
                elif 'First registration' in key: data['specifications']['year'] = val
                elif 'Fuel type' in key: data['specifications']['fuel_type'] = val
                elif 'Gearbox' in key or 'Transmission' in key: data['specifications']['transmission'] = val
                elif 'Power' in key: data['specifications']['power'] = val
                elif 'Seller' in key: data['seller_type'] = val

        # --- DETAILED SPECIFICATIONS (DataGrids) ---
        # This iterates through all definition lists (dl) in the page
        all_grids = soup.find_all('dl', class_=re.compile(r'DataGrid_defaultDlStyle'))

        for grid in all_grids:
            dts = grid.find_all('dt', class_=re.compile(r'DataGrid_defaultDtStyle'))
            dds = grid.find_all('dd', class_=re.compile(r'DataGrid_defaultDdStyle'))

            for dt, dd in zip(dts, dds):
                key = dt.get_text().strip()
                val = dd.get_text().strip()

                if not val or val == '-': continue

                k_lower = key.lower()
                specs = data['specifications'] # Short reference

                # Technical
                if ('engine size' in k_lower or 'displacement' in k_lower) and 'engine_size' not in specs:
                    specs['engine_size'] = val
                if 'cylinder' in k_lower and 'cylinders' not in specs:
                    specs['cylinders'] = val
                if 'gears' in k_lower and 'transmission' not in k_lower and 'gears' not in specs:
                    specs['gears'] = val
                if ('drive' in k_lower or 'drivetrain' in k_lower) and 'drive' not in specs:
                    specs['drive'] = val
                if 'door' in k_lower and 'doors' not in specs:
                    specs['doors'] = val
                if 'seat' in k_lower and 'seats' not in specs:
                    specs['seats'] = val

                # Type & Weight
                if 'body type' in k_lower and 'body_type' not in specs:
                    specs['body_type'] = val
                if 'vehicle type' in k_lower and 'vehicle_type' not in specs:
                    specs['vehicle_type'] = val
                if 'empty weight' in k_lower and 'empty_weight' not in specs:
                    specs['empty_weight'] = val
                if 'weight' in k_lower and 'empty' not in k_lower and 'weight' not in specs:
                    specs['weight'] = val

                # Environment
                if ('emission class' in k_lower or 'emission standard' in k_lower) and 'emission_class' not in specs:
                    specs['emission_class'] = val
                if 'co2' in k_lower and 'co2_emissions' not in specs:
                    specs['co2_emissions'] = val
                if 'consumption' in k_lower and 'fuel_consumption' not in specs:
                    specs['fuel_consumption'] = val

                # History & Paint
                if 'service history' in k_lower and 'full_service_history' not in specs:
                    specs['full_service_history'] = val
                if 'inspection' in k_lower and 'general_inspection' not in specs:
                     specs['general_inspection'] = val
                if ('colour' in k_lower or 'color' in k_lower) and 'manufacturer' not in k_lower and 'color' not in specs:
                    specs['color'] = val
                if 'paint' in k_lower and 'paint_type' not in specs:
                    specs['paint_type'] = val

                # ID
                if 'offer number' in k_lower and 'offer_number' not in specs:
                    specs['offer_number'] = val

        # --- EQUIPMENT / FEATURES ---
        data['features'] = []
        equip_section = soup.find('section', attrs={'data-cy': 'equipment-section'})

        if equip_section:
            dts = equip_section.find_all('dt', class_=re.compile(r'DataGrid_defaultDtStyle'))
            dds = equip_section.find_all('dd', class_=re.compile(r'DataGrid_defaultDdStyle'))

            for dt, dd in zip(dts, dds):
                cat = dt.get_text().strip()
                items = dd.find_all('li')
                for li in items:
                    txt = li.get_text().strip()
                    if txt:
                        data['features'].append({'category': cat, 'feature': txt})

        # --- SELLER ---
        data['seller'] = {}
        loc_link = soup.find('a', class_=re.compile(r'LocationWithPin_locationItem'))
        if loc_link:
            data['seller']['location'] = loc_link.get_text().strip()

        data['scrape_date'] = datetime.now().isoformat()
        data['source'] = 'autoscout24_detail_page'

        return data

    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return None

def scrape_batch(urls, max_cars=200, checkpoint_every=10):
    """Batched scraping with checkpoint system."""

    print(f"\n[CONFIG] Starting Details Scraping")
    print(f"         Available URLs: {len(urls)}")
    print(f"         Target: {max_cars}")
    print(f"         Checkpoint every: {checkpoint_every}")

    urls_to_scrape = urls[:max_cars]
    ckpt_file = os.path.join(checkpoint_folder, 'car_details_checkpoint.json')

    # Checkpoint Recovery
    all_details = []
    start_idx = 0

    if os.path.exists(ckpt_file):
        print(f"\n[CHECKPOINT] Found previous session")
        with open(ckpt_file, 'r', encoding='utf-8') as f:
            data = json.load(f)

        print(f"             Last index: {data['last_index']}")
        print(f"             Collected: {len(data['car_details'])}")

        if input("\nResume from checkpoint? (y/n): ").lower() == 'y':
            all_details = data['car_details']
            start_idx = data['last_index'] + 1
        else:
            print("Starting over...")

    session = create_session()

    # Logging
    log_file = os.path.join(logs_folder, f'details_log_{datetime.now().strftime("%Y%m%d_%H%M%S")}.txt')

    def log(msg):
        ts = datetime.now().strftime("%H:%M:%S")
        out = f"[{ts}] {msg}"
        print(out)
        with open(log_file, 'a', encoding='utf-8') as f:
            f.write(out + '\n')

    log(f"Starting from index {start_idx + 1}")
    log(f"Estimated time: {((len(urls_to_scrape) - start_idx) * 7.5) / 60:.1f} min")

    # Main Loop
    for i in range(start_idx, len(urls_to_scrape)):
        url = urls_to_scrape[i]

        log(f"\n{'='*40}")
        log(f"CAR {i + 1}/{len(urls_to_scrape)} | Collected: {len(all_details)}")
        log(f"URL: {url[:80]}...")

        try:
            detail = scrape_car_details(session, url)

            if detail:
                all_details.append(detail)
                log(f"      > ID: {detail.get('id', 'N/A')}")
                log(f"      > Title: {detail.get('title', 'N/A')}")
                log(f"      > Specs: {len(detail.get('specifications', {}))}")
                log(f"      > Status: Success")
            else:
                log(f"      > Status: Failed")

            # Save Checkpoint
            if (i + 1) % checkpoint_every == 0:
                with open(ckpt_file, 'w', encoding='utf-8') as f:
                    json.dump({
                        'last_index': i,
                        'last_update': datetime.now().isoformat(),
                        'car_details': all_details
                    }, f, ensure_ascii=False, indent=2)
                log(f"      [Checkpoint Saved]")

        except KeyboardInterrupt:
            log("\n[STOP] User interrupted.")
            with open(ckpt_file, 'w', encoding='utf-8') as f:
                json.dump({
                    'last_index': i,
                    'last_update': datetime.now().isoformat(),
                    'car_details': all_details
                }, f, ensure_ascii=False, indent=2)
            log("Progress saved. Exiting.")
            break

        except Exception as e:
            log(f"Error: {e}")
            continue

    # Cleanup Checkpoint
    if len(all_details) >= max_cars and os.path.exists(ckpt_file):
        os.remove(ckpt_file)
        log("Checkpoint removed (Task Complete).")

    return all_details

# ============================================
# EXECUTION
# ============================================

print("\n[START] Execution")
listings_path = os.path.join(data_folder, 'autoscout24_listings.json')

if os.path.exists(listings_path):
    with open(listings_path, 'r', encoding='utf-8') as f:
        listings = json.load(f).get('listings', [])

    urls = [l['link'] for l in listings if l.get('link')]
    print(f"Loaded {len(urls)} URLs from listings.")

    if not urls:
        print("[ERROR] No URLs found.")
    else:
        try:
            inp = input(f"How many cars to scrape? (Max {len(urls)}, Default 100): ").strip()
            max_cars = int(inp) if inp else 100
            max_cars = min(max_cars, len(urls))
        except:
            max_cars = 100

        final_data = scrape_batch(urls, max_cars=max_cars)

        # Statistics & Saving
        stats = {
            'total': len(final_data),
            'with_guid': sum(1 for d in final_data if d.get('id')),
            'avg_specs': sum(len(d.get('specifications', {})) for d in final_data) / len(final_data) if final_data else 0,
            'with_service_hist': sum(1 for d in final_data if d.get('specifications', {}).get('full_service_history'))
        }

        out_path = os.path.join(data_folder, 'autoscout24_car_details.json')
        print(f"\n[SAVE] Saving to: {out_path}")

        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump({
                'source': 'autoscout24',
                'date': datetime.now().isoformat(),
                'target_count': max_cars,
                'statistics': stats,
                'car_details': final_data
            }, f, ensure_ascii=False, indent=2)

        print("Success.")
        print(f"Stats: {stats}")

else:
    print(f"[ERROR] Listings file not found: {listings_path}")

Output streaming troncato alle ultime 5000 righe.
[19:55:06]       > Status: Success
[19:55:06] 
[19:55:06] CAR 3010/3633 | Collected: 2928
[19:55:06] URL: https://www.autoscout24.com/offers/peugeot-e-2008-707f0247-2b71-45c4-859d-4c2211...
[19:55:15]       > ID: 707f0247-2b71-45c4-859d-4c2211a2df80
[19:55:15]       > Title: Peugeot e-2008 50kWh Allure Pack
[19:55:15]       > Specs: 13
[19:55:15]       > Status: Success
[19:55:16]       [Checkpoint Saved]
[19:55:16] 
[19:55:16] CAR 3011/3633 | Collected: 2929
[19:55:16] URL: https://www.autoscout24.com/offers/volkswagen-tayron-124b20d0-4927-4b40-a779-02b...
[19:55:23]       > ID: 124b20d0-4927-4b40-a779-02b7677ad817
[19:55:23]       > Title: Volkswagen Tayron R-Line 2.0 TDI DSG 4Motion 360-Grad-Kamera
[19:55:23]       > Specs: 20
[19:55:23]       > Status: Success
[19:55:23] 
[19:55:23] CAR 3012/3633 | Collected: 2930
[19:55:23] URL: https://www.autoscout24.com/offers/suzuki-swift-b1707126-953f-428b-9431-177f79e4...
[19:55:32]       > I

# MERGING LISTING AND DETAILS

In [3]:
# ============================================================================
# DATA MERGING
# ============================================================================

import pandas as pd
import json
import os
from datetime import datetime

print("\n" + "="*70)
print("DATA MERGING")
print("="*70)

# Load data
listings_file = os.path.join(data_folder, 'autoscout24_listings.json')
details_file = os.path.join(data_folder, 'autoscout24_car_details.json')

with open(listings_file, 'r', encoding='utf-8') as f:
    listings_data = json.load(f).get('listings', [])

with open(details_file, 'r', encoding='utf-8') as f:
    car_details_data = json.load(f).get('car_details', [])

print(f"Loaded {len(listings_data)} listings, {len(car_details_data)} details")

# Convert to DataFrames
df_listings = pd.DataFrame(listings_data)
df_details = pd.DataFrame(car_details_data)

# Flatten specifications
if 'specifications' in df_details.columns:
    specs_df = pd.json_normalize(df_details['specifications'])
    specs_df.columns = ['spec_' + col for col in specs_df.columns]
    df_details = pd.concat([df_details.drop('specifications', axis=1), specs_df], axis=1)

# Binary encoding features
if 'features' in df_details.columns:
    # Extract all unique features
    all_features = set()
    for features_list in df_details['features']:
        if isinstance(features_list, list):
            for feat in features_list:
                feature_name = feat.get('feature', '')
                if feature_name:
                    all_features.add(feature_name)

    print(f"Encoding {len(all_features)} unique features as binary columns...")

    # Create binary columns dictionary
    feature_columns = {}
    for feature in sorted(all_features):
        col_name = 'feat_' + feature.lower().replace(' ', '_').replace('-', '_').replace('/', '_')
        col_name = ''.join(c for c in col_name if c.isalnum() or c == '_')

        feature_columns[col_name] = df_details['features'].apply(
            lambda x: 1 if isinstance(x, list) and any(f.get('feature') == feature for f in x) else 0
        )

    # Add count
    feature_columns['features_count'] = df_details['features'].apply(
        lambda x: len(x) if isinstance(x, list) else 0
    )

    # Concat all at once
    features_df = pd.DataFrame(feature_columns)
    df_details = pd.concat([df_details.drop('features', axis=1), features_df], axis=1)

# Flatten seller
if 'seller' in df_details.columns:
    seller_df = pd.json_normalize(df_details['seller'])
    seller_df.columns = ['seller_' + col for col in seller_df.columns]
    df_details = pd.concat([df_details.drop('seller', axis=1), seller_df], axis=1)

# Merge
combined_df = pd.merge(df_listings, df_details, on='id', how='inner', suffixes=('_listing', '_detail'))

print(f"Merged: {combined_df.shape[0]} rows × {combined_df.shape[1]} columns")

# Conflict resolution
priority_cols = ['title', 'price', 'year', 'fuel_type', 'seller_type', 'make', 'model']

for col in priority_cols:
    col_list = f'{col}_listing'
    col_det = f'{col}_detail'
    if col_list in combined_df.columns and col_det in combined_df.columns:
        combined_df[col] = combined_df[col_list].fillna(combined_df[col_det])
        combined_df.drop(columns=[col_list, col_det], inplace=True)

# Use raw numeric values
if 'price_raw' in combined_df.columns:
    combined_df['price'] = combined_df['price_raw']
if 'mileage_raw' in combined_df.columns:
    combined_df['mileage'] = combined_df['mileage_raw']

# Source field
if 'source_listing' in combined_df.columns:
    combined_df.drop(columns=['source_listing'], inplace=True)
if 'source_detail' in combined_df.columns:
    combined_df.drop(columns=['source_detail'], inplace=True)
combined_df['source'] = 'autoscout24'

# Remove unwanted columns
cols_to_remove = ['url', 'fuel_type_code', 'seller_name', 'link', 'page',
                  'scrape_date_listing', 'scrape_date_detail' , 'mileage_raw', 'price_raw']
for col in cols_to_remove:
    if col in combined_df.columns:
        combined_df.drop(columns=[col], inplace=True)

# Clean remaining _listing/_detail suffixes
rename_map = {}
for col in combined_df.columns:
    if col.endswith('_listing') or col.endswith('_detail'):
        new_col = col.replace('_listing', '').replace('_detail', '')
        rename_map[col] = new_col
if rename_map:
    combined_df.rename(columns=rename_map, inplace=True)

# Organize columns
priority_order = ['id', 'make', 'model', 'title', 'year', 'price',
                  'mileage',  'fuel_type', 'power', 'seller_type']
spec_cols = sorted([col for col in combined_df.columns if col.startswith('spec_')])
feat_cols = sorted([col for col in combined_df.columns if col.startswith('feat_')])
other_cols = [col for col in combined_df.columns
              if col not in priority_order and col not in spec_cols and col not in feat_cols]

final_order = [col for col in priority_order if col in combined_df.columns] + other_cols + spec_cols + feat_cols
combined_df = combined_df[final_order]

# Save
output_file = os.path.join(data_folder, 'autoscout24_combined_data.json')

combined_result = {
    'collection_date': datetime.now().isoformat(),
    'total_cars': len(combined_df),
    'combined_data': combined_df.to_dict(orient='records')
}

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(combined_result, f, ensure_ascii=False, indent=2)

print(f"\n[SAVE] {output_file}")
print(f"       {len(combined_df)} records, {len(combined_df.columns)} columns")
print("="*70)


DATA MERGING
Loaded 3633 listings, 3532 details
Encoding 144 unique features as binary columns...
Merged: 3532 rows × 193 columns

[SAVE] /content/drive/MyDrive/DataManagement_Project/data/autoscout24_combined_data.json
       3532 records, 180 columns
